# LLM-as-judge: hands-on (optional)
**LLMs, Animated · Lesson M4.4**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (plus a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print("ready")

## 1 · A simulated judge with position bias
We can't call a real LLM here, so we simulate one. Each answer has a hidden quality $q$. The judge sees two answers in slots 1 and 2 and picks slot 1 with probability

$$P(\text{slot 1 wins}) = \sigma(q_1 - q_2 + b), \qquad \sigma(x) = \frac{1}{1 + e^{-x}}$$

where $b > 0$ is a **position bias** toward the first slot. Implement it.

In [ ]:
def p_first_wins(q_first, q_second, bias):
    # TODO
    raise NotImplementedError("Exercise 1")

**Check 1.** An unbiased judge on equal answers is a coin flip. With $b = \ln(7/3)$ it picks the first slot 70% of the time.

In [ ]:
BIAS = np.log(7 / 3)
assert abs(p_first_wins(1.0, 1.0, 0.0) - 0.5) < 1e-12
assert abs(p_first_wins(1.0, 1.0, BIAS) - 0.7) < 1e-12
assert p_first_wins(2.0, 1.0, 0.0) > 0.5
print("✓ exercise 1")

## 2 · One order vs swap-and-average
Simulate 4,000 pairs where A and B are **equally good**. Write two scoring functions that return A's win rate:
- `win_rate_one_order`: A is always shown first.
- `win_rate_swapped`: judge each pair twice, A first and then B first, and give A the *average* of its two outcomes (1 = win, 0 = loss).

Use `rng.random(...) < p` to sample a verdict.

In [ ]:
def win_rate_one_order(qa, qb, bias, rng):
    # TODO: A in slot 1 every time
    raise NotImplementedError("Exercise 2a")

def win_rate_swapped(qa, qb, bias, rng):
    # TODO: order (A, B) and order (B, A); average A's two outcomes per pair
    raise NotImplementedError("Exercise 2b")

**Check 2**

In [ ]:
q = rng.normal(0, 1, 4000)
one, swap = win_rate_one_order(q, q, BIAS, rng), win_rate_swapped(q, q, BIAS, rng)
print(f"equal answers → A wins {one:.3f} with one order, {swap:.3f} with swap-and-average")
assert abs(one - 0.7) < 0.03 and abs(swap - 0.5) < 0.03
better = win_rate_swapped(q + 1.0, q, BIAS, rng)
assert better > 0.65, "when A really is better, swapping keeps it ahead"
print("✓ exercise 2")

## 3 · How consistent is the judge?
A common diagnostic: the fraction of pairs where the verdict *survives the swap* (the same answer wins in both orders). Implement it for one simulated run.

In [ ]:
def consistency_rate(qa, qb, bias, rng):
    # TODO: same winner in both orders → consistent
    raise NotImplementedError("Exercise 3")

**Check 3.** For equal answers, theory says consistency $= 2\,p(1-p)$ with $p = 0.7$, i.e. 0.42. A stronger bias makes the judge *less* consistent.

In [ ]:
cr = consistency_rate(q, q, BIAS, rng)
print("consistency on equal pairs:", round(cr, 3))
assert abs(cr - 2 * 0.7 * 0.3) < 0.03
assert consistency_rate(q, q, 2.0, rng) < consistency_rate(q, q, 0.5, rng)
print("✓ exercise 3")

## 4 · Cohen's kappa
To trust a judge, compare its labels with human labels on the same items. Implement

$$\kappa = \frac{p_o - p_e}{1 - p_e}, \qquad p_e = \sum_{\ell} P_{\text{judge}}(\ell)\,P_{\text{human}}(\ell)$$

for two arrays of labels (any number of label values).

In [ ]:
def cohens_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    # TODO: observed agreement, chance agreement from each rater's label rates, then kappa
    raise NotImplementedError("Exercise 4")

**Check 4.** The lesson's table (40 / 20 / 10 / 30) gives 0.40; a judge that says “pass” to everything gets 0.

In [ ]:
judge = np.array([1] * 40 + [1] * 20 + [0] * 10 + [0] * 30)
human = np.array([1] * 40 + [0] * 20 + [1] * 10 + [0] * 30)
assert abs(cohens_kappa(judge, human) - 0.40) < 1e-12
lazy = np.ones(100, dtype=int); humans = np.array([1] * 80 + [0] * 20)
assert abs(cohens_kappa(lazy, humans)) < 1e-12, "80% raw agreement, zero information"
assert abs(cohens_kappa(human, human) - 1) < 1e-12
print("✓ exercise 4")

## Experiment
Raw agreement vs kappa as labels get more lopsided. Simulate a judge that copies the human label 80% of the time and otherwise guesses with the same base rate. As the share of "good" answers grows, raw agreement climbs while kappa stays flat.

In [ ]:
rates = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
raw, kap = [], []
for r in rates:
    h = (rng.random(20000) < r).astype(int)
    guess = (rng.random(20000) < r).astype(int)
    j = np.where(rng.random(20000) < 0.8, h, guess)
    raw.append(np.mean(j == h)); kap.append(cohens_kappa(j, h))
plt.figure(figsize=(7, 4))
plt.plot(rates, raw, "o-", label="raw agreement"); plt.plot(rates, kap, "o-", label="Cohen's kappa")
plt.xlabel("share of answers humans label 'good'"); plt.ylim(0, 1.05); plt.legend()
plt.title("Same judge quality, lopsided labels"); plt.show()

## Recap
- A position-biased judge inflates whoever sits in slot 1; judging both orders and averaging removes it.
- Consistency under swapping is a cheap diagnostic of position bias.
- Validate a judge against human labels with Cohen's kappa, not raw agreement.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def p_first_wins(q_first, q_second, bias):
    return 1 / (1 + np.exp(-(q_first - q_second + bias)))

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def win_rate_one_order(qa, qb, bias, rng):
    a_wins = rng.random(len(qa)) < p_first_wins(qa, qb, bias)
    return a_wins.mean()

def win_rate_swapped(qa, qb, bias, rng):
    a_wins_1 = rng.random(len(qa)) < p_first_wins(qa, qb, bias)    # A in slot 1
    b_wins_2 = rng.random(len(qa)) < p_first_wins(qb, qa, bias)    # B in slot 1
    return ((a_wins_1.astype(float) + (~b_wins_2).astype(float)) / 2).mean()

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def consistency_rate(qa, qb, bias, rng):
    a_wins_1 = rng.random(len(qa)) < p_first_wins(qa, qb, bias)
    a_wins_2 = ~(rng.random(len(qa)) < p_first_wins(qb, qa, bias))
    return (a_wins_1 == a_wins_2).mean()

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def cohens_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    p_o = np.mean(a == b)
    labels = np.union1d(a, b)
    p_e = sum(np.mean(a == l) * np.mean(b == l) for l in labels)
    return (p_o - p_e) / (1 - p_e)